In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import ast
from sklearn.metrics import accuracy_score
import copy
from sklearn.metrics import (roc_auc_score, precision_score, recall_score, f1_score,average_precision_score)
from sklearn.neural_network import MLPRegressor

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import SplineTransformer, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.pipeline import make_pipeline
import matplotlib.pyplot as plt

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error,root_mean_squared_error
from scipy.stats import spearmanr
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
from scipy.spatial.distance import jensenshannon
from scipy.stats import wasserstein_distance

# Load

In [ ]:
newness = **Newness Data**
surprise = **Surprise Data**
value = ** Value Data**

In [ ]:
BASELINE_YEAR=1600

In [ ]:
gold_set_70 = **selected evaluation set**
creative_70 = ** creativity score**
artnet_2024 = ** tabular data**
merged_df = pd.merge(gold_set_70, artnet_2024[["artwork id","artist id","year born","workyear modifier","price","artist",'est lo usd','est hi usd', 'sale price usd']], on='artwork id', how='left')

In [ ]:
artist_list = ["Pablo Picasso","Pierre-Auguste Renoir","Käthe Kollwitz"]
keep_list = merged_df["artist"].isin(artist_list)
merged_df = merged_df[keep_list]
creative_70 = creative_70[keep_list]

In [ ]:
newness_df = np.zeros(merged_df.shape[0], dtype=object)
surprise_df = np.zeros(merged_df.shape[0], dtype=object)
value_df = np.zeros(merged_df.shape[0], dtype=object)
for i in range(merged_df.shape[0]):
    row_index = merged_df.index[i]
    newness_df[i] =newness[row_index]
    surprise_df[i] =surprise[row_index]
    value_df[i] = value[row_index]

In [ ]:
def logit(p, epsilon=1e-10):
    # Clip p to avoid log(0) or log(1/0)
    p = np.clip(p, epsilon, 1 - epsilon)
    return np.log(p / (1 - p))

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [ ]:
def topk_hit_rate_curve(y_true, y_pred, rates):
    hit_rates = []
    n = len(y_true)

    for rate in rates:
        k = int(n * rate)
        if k == 0:
            hit_rates.append(np.nan)
            continue

        true_top_idx = np.argsort(y_true)[-k:]
        pred_top_idx = np.argsort(y_pred)[-k:]

        hit_rate = len(set(true_top_idx) & set(pred_top_idx)) / k
        hit_rates.append(hit_rate)

    return hit_rates

In [ ]:
def kl_divergence(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)

    # Normalize (important if they are not exact probabilities)
    p = p / p.sum()
    q = q / q.sum()

    # Avoid log(0)
    p = np.clip(p, eps, 1)
    q = np.clip(q, eps, 1)

    return np.sum(p * np.log(p / q))

# Basic

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error,root_mean_squared_error
from scipy.stats import spearmanr

In [ ]:
epsilon=1e-6

In [ ]:
newness_min=np.min(newness_df)
newness_max = np.max(newness_df)
newness_normalized=(newness_df - newness_min)/(newness_max-newness_min)

In [ ]:
surprise_min=np.min(surprise_df)
surprise_max = np.max(surprise_df)
surprise_normalized=(surprise_df - surprise_min)/(surprise_max-surprise_min)

In [ ]:
value_min=np.min(value_df)
value_max = np.max(value_df)
value_normalized=(value_df - value_min)/(value_max-value_min)

In [ ]:
creativity_min = np.min(creative_70)
creativity_max =np.max(creative_70)
creativity_normalized = (creative_70 - creativity_min)/(creativity_max-creativity_min)

In [ ]:
y = np.log((creativity_normalized+epsilon).astype(float))
X = np.column_stack([
    np.log((newness_normalized+epsilon).astype(float)),
    np.log((surprise_normalized+epsilon).astype(float)),
    np.log((value_normalized+epsilon).astype(float))
])

In [ ]:
kf = KFold(n_splits=5,shuffle=True,random_state=24)
kf.get_n_splits()
mses=[]
maes=[]
rmses=[]
mapes=[]

kls=[]
jss=[]
wds=[]
for i, (train_index, test_index) in enumerate(kf.split(X)):
    X_train = X[train_index]
    X_test = X[test_index]
    y_train = y[train_index]
    y_test = y[test_index]

    model = LinearRegression()
    model.fit(X_train, y_train)
    
    true_label = sigmoid(y_test)
    pred_y = sigmoid(model.predict(X_test))

    mse = mean_squared_error(true_label, np.array(pred_y))
    # print(f"Test Linear Regression model mse: {mse:.4f}")
    mae = mean_absolute_error(true_label, pred_y)
    # print("Test mae:", mae)
    rmse = root_mean_squared_error(true_label, pred_y)
    # print("Test RMSE:", RMSE)
    mape = mean_absolute_percentage_error(true_label, pred_y)
    
    kl = kl_divergence(true_label, pred_y)
    # print("Test KL:", kl )
    js = jensenshannon(true_label, pred_y)**2
    # print("Test JS:", js )
    wd = wasserstein_distance(true_label, pred_y)
    # print("Test Wasserstein Distance :", wd )

    mses.append(mse)
    maes.append(mae)
    rmses.append(rmse)
    mapes.append(mape)
    
    kls.append(kl)
    jss.append(js)
    wds.append(wd)

print(f"Average mse is {np.mean(mses)}, std is {np.std(mses)}")
print(f"Average mae is {np.mean(maes)}, std is {np.std(maes)}")
print(f"Average rmse is {np.mean(rmses)}, std is {np.std(rmses)}")
print(f"Average mape is {np.mean(mapes)}, std is {np.std(mapes)}")
print("-------------------------------------------------------")
print(f"Average KL is {np.mean(kls)}, std is {np.std(kls)}")
print(f"Average JS is {np.mean(jss)}, std is {np.std(jss)}")
print(f"Average WD is {np.mean(wds)}, std is {np.std(wds)}")

In [ ]:
import copy
basic_mae = copy.deepcopy(maes)
basic_rmse = copy.deepcopy(rmses)
basic_js = copy.deepcopy(jss)
basic_wd = copy.deepcopy(wds)

# Splines

In [ ]:
surprise_min=np.min(surprise_df)
surprise_max = np.max(surprise_df)
surprise_normalized=(surprise_df - surprise_min)/(surprise_max-surprise_min)
newness_min=np.min(newness_df)
newness_max = np.max(newness_df)
newness_normalized=(newness_df - newness_min)/(newness_max-newness_min)
value_min=np.min(value_df)
value_max = np.max(value_df)
value_normalized=(value_df - value_min)/(value_max-value_min)
creativity_min = np.min(creative_70)
creativity_max =np.max(creative_70)
creativity_normalized = (creative_70 - creativity_min)/(creativity_max-creativity_min)

In [ ]:
y = np.log((creativity_normalized+epsilon).astype(float))
X = np.column_stack([
    np.log((newness_normalized+epsilon).astype(float)),
    np.log((surprise_normalized+epsilon).astype(float)),
    np.log((value_normalized+epsilon).astype(float))
])

In [ ]:
preprocess = ColumnTransformer([
    ("spline", SplineTransformer(
        degree=4,           # cubic
        n_knots=8,          # tune with CV (e.g., 5–10 common)
        extrapolation="linear",
        include_bias=False  # avoid intercept duplication
    ), [1,2]),
], remainder="drop")

In [ ]:
kf = KFold(n_splits=5,shuffle=True,random_state=24)
kf.get_n_splits()
mses=[]
maes=[]
rmses=[]
mapes=[]

kls=[]
jss=[]
wds=[]
for i, (train_index, test_index) in enumerate(kf.split(X)):
    X_train = X[train_index]
    X_test = X[test_index]
    y_train = y[train_index]
    y_test = y[test_index]

    model = make_pipeline(
        preprocess,
        LinearRegression()
    )
    model.fit(X_train, y_train)
    
    true_label = sigmoid(y_test)
    pred_y = sigmoid(model.predict(X_test))

    mse = mean_squared_error(true_label, np.array(pred_y))
    # print(f"Test Linear Regression model mse: {mse:.4f}")
    mae = mean_absolute_error(true_label, pred_y)
    # print("Test mae:", mae)
    rmse = root_mean_squared_error(true_label, pred_y)
    # print("Test RMSE:", RMSE)
    mape = mean_absolute_percentage_error(true_label, pred_y)

    kl = kl_divergence(true_label, pred_y)
    # print("Test KL:", kl )
    js = jensenshannon(true_label, pred_y)**2
    # print("Test JS:", js )
    wd = wasserstein_distance(true_label, pred_y)
    # print("Test Wasserstein Distance :", wd )

    mses.append(mse)
    maes.append(mae)
    rmses.append(rmse)
    mapes.append(mape)

    kls.append(kl)
    jss.append(js)
    wds.append(wd)

print(f"Average mse is {np.mean(mses)}, std is {np.std(mses)}")
print(f"Average mae is {np.mean(maes)}, std is {np.std(maes)}")
print(f"Average rmse is {np.mean(rmses)}, std is {np.std(rmses)}")
print(f"Average mape is {np.mean(mapes)}, std is {np.std(mapes)}")
print("-------------------------------------------------------")
print(f"Average KL is {np.mean(kls)}, std is {np.std(kls)}")
print(f"Average JS is {np.mean(jss)}, std is {np.std(jss)}")
print(f"Average WD is {np.mean(wds)}, std is {np.std(wds)}")

Check Significants:

In [ ]:
from scipy.stats import ttest_rel, ttest_ind

In [ ]:
print(ttest_ind(basic_mae, maes))
print(ttest_ind(basic_rmse, rmses))
print(ttest_ind(basic_js, jss))
print(ttest_ind(basic_wd, wds))

In [ ]:
print((np.mean(basic_mae) - np.mean(maes))/np.mean(maes))
print((np.mean(basic_rmse) - np.mean(rmses))/np.mean(rmses))
print((np.mean(basic_js) - np.mean(jss))/np.mean(jss))
print((np.mean(wds)-np.mean(basic_wd))/np.mean(basic_wd))